In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("=" * 50)
print("Behavioral Anomaly Detection VAE - V2")
print("R26-IT-059 | IT22215710 | Karunarathne D C")
print("=" * 50)
print("\nAll libraries imported successfully!")

Behavioral Anomaly Detection VAE - V2
R26-IT-059 | IT22215710 | Karunarathne D C

All libraries imported successfully!


In [2]:
# Define data path
DATA_PATH = "../data/raw/"

# Load all tables
student_info = pd.read_csv(DATA_PATH + "studentInfo.csv")
student_vle = pd.read_csv(DATA_PATH + "studentVle.csv")
student_assessment = pd.read_csv(DATA_PATH + "studentAssessment.csv")
assessments = pd.read_csv(DATA_PATH + "assessments.csv")
student_registration = pd.read_csv(DATA_PATH + "studentRegistration.csv")
vle = pd.read_csv(DATA_PATH + "vle.csv")
courses = pd.read_csv(DATA_PATH + "courses.csv")

# Create at-risk labels
student_info['at_risk'] = (
    student_info['final_result'] == 'Withdrawn'
).astype(int)

print("All tables loaded!")
print(f"\nstudentInfo:         {student_info.shape}")
print(f"studentVle:          {student_vle.shape}")
print(f"studentAssessment:   {student_assessment.shape}")
print(f"assessments:         {assessments.shape}")
print(f"studentRegistration: {student_registration.shape}")
print(f"vle:                 {vle.shape}")
print(f"courses:             {courses.shape}")
print(f"\nAt-risk rate: {student_info['at_risk'].mean()*100:.1f}%")
print(f"\nModules available:")
print(student_info['code_module'].unique())

All tables loaded!

studentInfo:         (32593, 13)
studentVle:          (10655280, 6)
studentAssessment:   (173912, 5)
assessments:         (206, 6)
studentRegistration: (32593, 5)
vle:                 (6364, 6)
courses:             (22, 3)

At-risk rate: 31.2%

Modules available:
['AAA' 'BBB' 'CCC' 'DDD' 'EEE' 'FFF' 'GGG']


In [3]:
# Explore each module
print("=" * 50)
print("MODULE ANALYSIS")
print("=" * 50)

module_stats = student_info.groupby('code_module').agg(
    total_students=('id_student', 'count'),
    at_risk_count=('at_risk', 'sum'),
    at_risk_rate=('at_risk', 'mean')
).reset_index()

module_stats['at_risk_rate'] = (
    module_stats['at_risk_rate'] * 100
).round(1)

print(module_stats.to_string(index=False))

# Course lengths
print(f"\nCourse lengths (days):")
print(courses[['code_module', 'length']].drop_duplicates())

# Assessments per module
print(f"\nAssessments per module:")
print(assessments.groupby('code_module')[
    'id_assessment'
].count().reset_index().rename(
    columns={'id_assessment': 'assessment_count'}
))

MODULE ANALYSIS
code_module  total_students  at_risk_count  at_risk_rate
        AAA             748            126          16.8
        BBB            7909           2388          30.2
        CCC            4434           1975          44.5
        DDD            6272           2250          35.9
        EEE            2934            722          24.6
        FFF            7762           2403          31.0
        GGG            2534            292          11.5

Course lengths (days):


KeyError: "['length'] not in index"

In [4]:
# Check courses columns
print(courses.columns.tolist())
print(courses.head())

['code_module', 'code_presentation', 'module_presentation_length']
  code_module code_presentation  module_presentation_length
0         AAA             2013J                         268
1         AAA             2014J                         269
2         BBB             2013J                         268
3         BBB             2014J                         262
4         BBB             2013B                         240


In [5]:
# Module analysis
print("=" * 50)
print("MODULE ANALYSIS")
print("=" * 50)

module_stats = student_info.groupby('code_module').agg(
    total_students=('id_student', 'count'),
    at_risk_count=('at_risk', 'sum'),
    at_risk_rate=('at_risk', 'mean')
).reset_index()

module_stats['at_risk_rate'] = (
    module_stats['at_risk_rate'] * 100
).round(1)

print(module_stats.to_string(index=False))

# Course lengths
print(f"\nCourse lengths (days):")
print(courses.groupby('code_module')[
    'module_presentation_length'
].mean().reset_index().round(0))

# Assessments per module
print(f"\nAssessments per module:")
print(assessments.groupby('code_module')[
    'id_assessment'
].count().reset_index().rename(
    columns={'id_assessment': 'assessment_count'}
))

MODULE ANALYSIS
code_module  total_students  at_risk_count  at_risk_rate
        AAA             748            126          16.8
        BBB            7909           2388          30.2
        CCC            4434           1975          44.5
        DDD            6272           2250          35.9
        EEE            2934            722          24.6
        FFF            7762           2403          31.0
        GGG            2534            292          11.5

Course lengths (days):
  code_module  module_presentation_length
0         AAA                       268.0
1         BBB                       251.0
2         CCC                       255.0
3         DDD                       251.0
4         EEE                       259.0
5         FFF                       254.0
6         GGG                       257.0

Assessments per module:
  code_module  assessment_count
0         AAA                12
1         BBB                42
2         CCC                20
3         DDD  

In [6]:
# ============================================
# V2 KEY FEATURE: Academic Calendar
# Admin manually defines semester structure
# ============================================

print("Building Academic Calendar...")

# Define academic calendar per module
# Based on OULAD course structure
# Days converted to weeks

academic_calendar = {
    'AAA': {
        'total_weeks': 38,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [34, 35, 36, 37],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [33, 34]
    },
    'BBB': {
        'total_weeks': 36,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [32, 33, 34, 35],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [31, 32]
    },
    'CCC': {
        'total_weeks': 36,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [32, 33, 34, 35],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [31, 32]
    },
    'DDD': {
        'total_weeks': 36,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [32, 33, 34, 35],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [31, 32]
    },
    'EEE': {
        'total_weeks': 37,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [33, 34, 35, 36],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [32, 33]
    },
    'FFF': {
        'total_weeks': 36,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [32, 33, 34, 35],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [31, 32]
    },
    'GGG': {
        'total_weeks': 37,
        'baseline_weeks': [0, 1, 2],
        'exam_weeks': [33, 34, 35, 36],
        'holiday_weeks': [8, 9],
        'high_activity_weeks': [32, 33]
    }
}

print("Academic calendar defined!")
print(f"\nModules configured: {list(academic_calendar.keys())}")
print(f"\nExample - Module CCC:")
print(f"  Total weeks:    {academic_calendar['CCC']['total_weeks']}")
print(f"  Baseline weeks: {academic_calendar['CCC']['baseline_weeks']}")
print(f"  Holiday weeks:  {academic_calendar['CCC']['holiday_weeks']}")
print(f"  Exam weeks:     {academic_calendar['CCC']['exam_weeks']}")

Building Academic Calendar...
Academic calendar defined!

Modules configured: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']

Example - Module CCC:
  Total weeks:    36
  Baseline weeks: [0, 1, 2]
  Holiday weeks:  [8, 9]
  Exam weeks:     [32, 33, 34, 35]


In [7]:
# ============================================
# V2 KEY FEATURE: Curriculum Baseline
# Expected behavior per module per week
# Based on actual OULAD resource counts
# ============================================

print("Building Curriculum Baseline...")

# Count actual resources per module per week
# This tells us what a normal student SHOULD access

# Merge VLE with activity types
vle_merged = student_vle.merge(
    vle[['id_site', 'activity_type']],
    on='id_site',
    how='left'
)

# Convert date to week
vle_merged['week'] = (vle_merged['date'] // 7).clip(lower=0)

# Count resources available per module per week
resources_per_week = vle_merged[
    vle_merged['activity_type'].isin([
        'resource', 'oucontent', 'url'
    ])
].groupby(
    ['code_module', 'week']
)['id_site'].nunique().reset_index()

resources_per_week.columns = [
    'code_module', 'week', 'resources_available'
]

# Count assessments per module per week
assessments_merged = assessments.copy()
assessments_merged['week'] = (
    assessments_merged['date'] // 7
).clip(lower=0)

assessments_per_week = assessments_merged.groupby(
    ['code_module', 'week']
)['id_assessment'].count().reset_index()
assessments_per_week.columns = [
    'code_module', 'week', 'assessments_due'
]

print("Curriculum baseline built!")
print(f"\nResources per week sample:")
print(resources_per_week.head(10))
print(f"\nAssessments per week sample:")
print(assessments_per_week[
    assessments_per_week['assessments_due'] > 0
].head(10))

Building Curriculum Baseline...
Curriculum baseline built!

Resources per week sample:
  code_module  week  resources_available
0         AAA     0                  252
1         AAA     1                  145
2         AAA     2                  163
3         AAA     3                  141
4         AAA     4                  141
5         AAA     5                  137
6         AAA     6                  128
7         AAA     7                  193
8         AAA     8                  127
9         AAA     9                  121

Assessments per week sample:
  code_module  week  assessments_due
0         AAA   2.0                2
1         AAA   7.0                2
2         AAA  16.0                2
3         AAA  23.0                2
4         AAA  30.0                2
5         BBB   1.0                1
6         BBB   2.0                3
7         BBB   5.0                1
8         BBB   6.0                3
9         BBB   7.0                3


In [8]:
# ============================================
# V2 KEY FEATURE: Expected Behavior Model
# What a normal engaged student should do
# per module per week
# ============================================

print("Building Expected Behavior Model...")

# Calculate expected clicks per module per week
# Based on average of TOP 25% engaged students
# (not all students - use only highly engaged)

# Get top 25% engaged students per module
total_clicks_per_student = student_vle.groupby(
    ['code_module', 'code_presentation', 'id_student']
)['sum_click'].sum().reset_index()

# Per module get 75th percentile threshold
module_engagement_threshold = total_clicks_per_student.groupby(
    'code_module'
)['sum_click'].quantile(0.75).reset_index()
module_engagement_threshold.columns = [
    'code_module', 'engagement_threshold'
]

# Get highly engaged students
engaged_students = total_clicks_per_student.merge(
    module_engagement_threshold,
    on='code_module'
)
engaged_students = engaged_students[
    engaged_students['sum_click'] >= 
    engaged_students['engagement_threshold']
]

print(f"Highly engaged students identified:")
print(engaged_students.groupby(
    'code_module'
)['id_student'].count())

# Calculate expected weekly clicks from engaged students
engaged_vle = vle_merged.merge(
    engaged_students[['code_module', 
                      'code_presentation', 
                      'id_student']],
    on=['code_module', 'code_presentation', 'id_student'],
    how='inner'
)

expected_behavior = engaged_vle.groupby(
    ['code_module', 'week']
).agg(
    expected_clicks=('sum_click', 'median'),
    expected_active_days=('date', 'nunique'),
    expected_resource_views=('id_site', 'nunique')
).reset_index()

print(f"\nExpected behavior per module per week:")
print(expected_behavior[
    expected_behavior['code_module'] == 'CCC'
].head(10))

Building Expected Behavior Model...
Highly engaged students identified:
code_module
AAA     184
BBB    1656
CCC     996
DDD    1439
EEE     672
FFF    1773
GGG     592
Name: id_student, dtype: int64

Expected behavior per module per week:
   code_module  week  expected_clicks  expected_active_days  \
78         CCC     0              1.0                    25   
79         CCC     1              2.0                     7   
80         CCC     2              2.0                     7   
81         CCC     3              2.0                     7   
82         CCC     4              2.0                     7   
83         CCC     5              2.0                     7   
84         CCC     6              2.0                     7   
85         CCC     7              2.0                     7   
86         CCC     8              2.0                     7   
87         CCC     9              2.0                     7   

    expected_resource_views  
78                      231  
79     

In [9]:
# Fix expected behavior using mean instead of median
# and per student per week aggregation first

print("Fixing expected behavior calculation...")

# Step 1: aggregate per student per week first
engaged_weekly = engaged_vle.groupby(
    ['code_module', 'id_student', 'week']
).agg(
    weekly_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique')
).reset_index()

# Step 2: then average across engaged students
expected_behavior_fixed = engaged_weekly.groupby(
    ['code_module', 'week']
).agg(
    expected_clicks=('weekly_clicks', 'mean'),
    expected_active_days=('active_days', 'mean')
).reset_index().round(1)

print("Expected behavior fixed!")
print(f"\nModule CCC expected behavior:")
print(expected_behavior_fixed[
    expected_behavior_fixed['code_module'] == 'CCC'
].head(10).to_string(index=False))

# Save expected behavior
expected_behavior_fixed.to_csv(
    '../results/metrics/expected_behavior.csv',
    index=False
)
print(f"\nExpected behavior saved!")
print(f"Total records: {len(expected_behavior_fixed):,}")

Fixing expected behavior calculation...
Expected behavior fixed!

Module CCC expected behavior:
code_module  week  expected_clicks  expected_active_days
        CCC     0            275.9                  10.6
        CCC     1            154.3                   4.1
        CCC     2            175.5                   4.4
        CCC     3            103.0                   4.0
        CCC     4            107.1                   4.4
        CCC     5             79.2                   4.0
        CCC     6             84.4                   3.7
        CCC     7             84.0                   3.4
        CCC     8            101.0                   3.6
        CCC     9            150.5                   4.1

Expected behavior saved!
Total records: 272


In [10]:
# ============================================
# V2 FEATURE ENGINEERING
# Better features with curriculum awareness
# ============================================

print("Building V2 Features...")

# Merge all VLE data
student_vle_merged = student_vle.merge(
    vle[['id_site', 'activity_type']],
    on='id_site', how='left'
)
student_vle_merged['week'] = (
    student_vle_merged['date'] // 7
).clip(lower=0)

# Weekly aggregation per student per module
print("Aggregating weekly features...")

weekly_base = student_vle_merged.groupby(
    ['id_student', 'code_module', 
     'code_presentation', 'week']
).agg(
    weekly_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    content_diversity=('activity_type', 'nunique')
).reset_index()

# Forum activity
forum = student_vle_merged[
    student_vle_merged['activity_type'] == 'forumng'
].groupby(
    ['id_student', 'code_module', 
     'code_presentation', 'week']
)['sum_click'].sum().reset_index()
forum.columns = [
    'id_student', 'code_module',
    'code_presentation', 'week', 'forum_clicks'
]

# Resource access
resource = student_vle_merged[
    student_vle_merged['activity_type'].isin([
        'resource', 'oucontent', 'url'
    ])
].groupby(
    ['id_student', 'code_module',
     'code_presentation', 'week']
)['sum_click'].sum().reset_index()
resource.columns = [
    'id_student', 'code_module',
    'code_presentation', 'week', 'resource_clicks'
]

# Quiz activity
quiz = student_vle_merged[
    student_vle_merged['activity_type'] == 'quiz'
].groupby(
    ['id_student', 'code_module',
     'code_presentation', 'week']
)['sum_click'].sum().reset_index()
quiz.columns = [
    'id_student', 'code_module',
    'code_presentation', 'week', 'quiz_clicks'
]

# Merge all features
features_v2 = weekly_base.copy()
features_v2 = features_v2.merge(
    forum, on=['id_student', 'code_module',
               'code_presentation', 'week'],
    how='left'
)
features_v2 = features_v2.merge(
    resource, on=['id_student', 'code_module',
                  'code_presentation', 'week'],
    how='left'
)
features_v2 = features_v2.merge(
    quiz, on=['id_student', 'code_module',
              'code_presentation', 'week'],
    how='left'
)
features_v2 = features_v2.fillna(0)

print(f"Base features shape: {features_v2.shape}")
print(f"\nColumns: {list(features_v2.columns)}")

Building V2 Features...
Aggregating weekly features...
Base features shape: (582772, 10)

Columns: ['id_student', 'code_module', 'code_presentation', 'week', 'weekly_clicks', 'active_days', 'content_diversity', 'forum_clicks', 'resource_clicks', 'quiz_clicks']


In [13]:
# Add submission latency features
print("Adding submission features...")

# Merge assessments with student submissions
submission_data = student_assessment.merge(
    assessments[['id_assessment', 'code_module',
                 'code_presentation', 'date',
                 'assessment_type']],
    on='id_assessment',
    how='left'
)

# Calculate submission latency
submission_data['submission_latency'] = (
    submission_data['date_submitted'] -
    submission_data['date']
)

# Convert to week - handle NaN properly
submission_data['week'] = (
    submission_data['date'] // 7
).clip(lower=0).fillna(0).astype(int)

# Aggregate per student per module per week
submission_weekly = submission_data.groupby(
    ['id_student', 'code_module',
     'code_presentation', 'week']
).agg(
    avg_submission_latency=('submission_latency', 'mean'),
    late_submissions=('submission_latency',
                     lambda x: (x > 0).sum()),
    missing_submissions=('score',
                        lambda x: x.isna().sum()),
    avg_score=('score', 'mean')
).reset_index()

# Merge with main features
features_v2 = features_v2.merge(
    submission_weekly,
    on=['id_student', 'code_module',
        'code_presentation', 'week'],
    how='left'
)

features_v2['avg_submission_latency'] = (
    features_v2['avg_submission_latency'].fillna(0)
)
features_v2['late_submissions'] = (
    features_v2['late_submissions'].fillna(0)
)
features_v2['missing_submissions'] = (
    features_v2['missing_submissions'].fillna(0)
)
features_v2['avg_score'] = (
    features_v2['avg_score'].fillna(
        features_v2['avg_score'].median()
    )
)

print(f"Features with submissions: {features_v2.shape}")
print(f"\nNew columns added:")
print(f"  avg_submission_latency")
print(f"  late_submissions")
print(f"  missing_submissions")
print(f"  avg_score")
print(f"\nSample submission latency stats:")
print(features_v2['avg_submission_latency'].describe())

Adding submission features...
Features with submissions: (582772, 14)

New columns added:
  avg_submission_latency
  late_submissions
  missing_submissions
  avg_score

Sample submission latency stats:
count    582772.000000
mean         -0.607687
std           7.782172
min        -237.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         372.000000
Name: avg_submission_latency, dtype: float64


In [14]:
# ============================================
# V2 KEY FEATURE: Curriculum Compliance Score
# How much is student doing vs expected?
# ============================================

print("Adding curriculum compliance scores...")

# Merge with expected behavior
features_v2 = features_v2.merge(
    expected_behavior_fixed[['code_module', 'week',
                              'expected_clicks',
                              'expected_active_days']],
    on=['code_module', 'week'],
    how='left'
)

# Fill missing expected values with module average
features_v2['expected_clicks'] = features_v2[
    'expected_clicks'
].fillna(features_v2.groupby(
    'code_module'
)['expected_clicks'].transform('mean'))

features_v2['expected_active_days'] = features_v2[
    'expected_active_days'
].fillna(features_v2.groupby(
    'code_module'
)['expected_active_days'].transform('mean'))

# Calculate compliance ratios
# 1.0 = meeting expectations
# < 1.0 = below expectations (concerning)
# > 1.0 = above expectations (engaged)

features_v2['click_compliance'] = (
    features_v2['weekly_clicks'] /
    (features_v2['expected_clicks'] + 1)
).clip(upper=2.0)

features_v2['day_compliance'] = (
    features_v2['active_days'] /
    (features_v2['expected_active_days'] + 1)
).clip(upper=2.0)

# Overall compliance score
features_v2['curriculum_compliance'] = (
    features_v2['click_compliance'] * 0.5 +
    features_v2['day_compliance'] * 0.5
)

print(f"Shape: {features_v2.shape}")
print(f"\nCurriculum compliance stats:")
print(features_v2['curriculum_compliance'].describe())
print(f"\nMean compliance by at_risk:")

# Add at_risk label
features_v2 = features_v2.merge(
    student_info[['id_student', 'code_module',
                  'code_presentation', 'at_risk']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

print(features_v2.groupby('at_risk')[
    'curriculum_compliance'
].mean())

Adding curriculum compliance scores...
Shape: (582772, 19)

Curriculum compliance stats:
count    582772.000000
mean          0.581729
std           0.424119
min           0.032554
25%           0.237883
50%           0.468584
75%           0.819638
max           2.000000
Name: curriculum_compliance, dtype: float64

Mean compliance by at_risk:
at_risk
0    0.59657
1    0.44244
Name: curriculum_compliance, dtype: float64


In [15]:
# ============================================
# V2 KEY FEATURE: Calendar Awareness
# Mark holiday and exam weeks
# Don't flag anomalies during breaks!
# ============================================

print("Adding calendar awareness...")

def get_week_type(row):
    module = row['code_module']
    week = row['week']
    
    if module not in academic_calendar:
        return 'normal'
    
    cal = academic_calendar[module]
    
    if week in cal['holiday_weeks']:
        return 'holiday'
    elif week in cal['exam_weeks']:
        return 'exam'
    elif week in cal['high_activity_weeks']:
        return 'pre_exam'
    elif week in cal['baseline_weeks']:
        return 'baseline'
    else:
        return 'normal'

features_v2['week_type'] = features_v2.apply(
    get_week_type, axis=1
)

print(f"Week types distribution:")
print(features_v2['week_type'].value_counts())

print(f"\nMean compliance by week type:")
print(features_v2.groupby('week_type')[
    'curriculum_compliance'
].mean().round(3))

Adding calendar awareness...
Week types distribution:
week_type
normal      425076
baseline     73982
holiday      35949
exam         34882
pre_exam     12883
Name: count, dtype: int64

Mean compliance by week type:
week_type
baseline    0.551
exam        0.614
holiday     0.565
normal      0.584
pre_exam    0.629
Name: curriculum_compliance, dtype: float64


In [16]:
# Filter to monitoring weeks only
# Exclude holiday weeks from anomaly detection
# Keep first 17 weeks for consistency

print("Filtering and saving V2 features...")

# Keep only non-holiday weeks for anomaly detection
features_v2_clean = features_v2[
    (features_v2['week'] < 17) &
    (features_v2['week_type'] != 'holiday')
].copy()

# Define V2 feature columns
FEATURE_COLS_V2 = [
    'weekly_clicks',
    'active_days',
    'content_diversity',
    'forum_clicks',
    'resource_clicks',
    'quiz_clicks',
    'avg_submission_latency',
    'late_submissions',
    'curriculum_compliance',
    'click_compliance'
]

print(f"V2 Features: {len(FEATURE_COLS_V2)}")
for i, f in enumerate(FEATURE_COLS_V2):
    print(f"  F{i+1}: {f}")

print(f"\nShape after filtering: {features_v2_clean.shape}")
print(f"Unique students: {features_v2_clean['id_student'].nunique():,}")
print(f"Week types remaining:")
print(features_v2_clean['week_type'].value_counts())

# Save
features_v2_clean.to_csv(
    '../results/metrics/features_v2.csv',
    index=False
)
print(f"\nV2 features saved!")

Filtering and saving V2 features...
V2 Features: 10
  F1: weekly_clicks
  F2: active_days
  F3: content_diversity
  F4: forum_clicks
  F5: resource_clicks
  F6: quiz_clicks
  F7: avg_submission_latency
  F8: late_submissions
  F9: curriculum_compliance
  F10: click_compliance

Shape after filtering: (288516, 21)
Unique students: 26,035
Week types remaining:
week_type
normal      214534
baseline     73982
Name: count, dtype: int64

V2 features saved!
